# 모듈 03: LCEL - | 파이프 연산자

모듈 02에서 프롬프트를 채우고 → llm에 전달하는 두 단계를 직접 연결했습니다.
이번에는 **`|` 파이프 연산자 하나**로 여러 컴포넌트를 한 줄로 연결합니다.

---

## 이 노트북에서 할 것

| 단계 | 내용 |
|------|------|
| 1 | LCEL 기본 체인: prompt \| llm \| parser 구성 |
| 2 | invoke() / stream() / batch() 세 가지 실행 방식 |
| 3 | RunnablePassthrough - 원본 입력을 체인 끝까지 보존하기 |
| 4 | RunnableParallel - 여러 체인을 동시에 실행하기 |
| 5 | RunnableLambda - 파이프라인 중간에 커스텀 함수 끼워넣기 |

**예상 소요 시간**: 약 60분

> **전제 조건**: 모듈 02가 완료되어 `.env` 파일에 `GOOGLE_API_KEY`가 설정되어 있어야 합니다.

## 학습 목표

이 모듈을 완료하면 다음 세 가지를 할 수 있습니다:

1. **`|` 연산자로 체인 구성하기**: `prompt | llm | parser` 문법으로 컴포넌트를 연결하고, `invoke()` / `stream()` / `batch()` 세 가지 방식으로 실행하는 법을 압니다.

2. **RunnablePassthrough + RunnableParallel 활용하기**: 원본 입력을 보존하면서 여러 체인을 동시에 실행해 딕셔너리 형태로 결과를 받는 법을 압니다.

3. **RunnableLambda로 커스텀 함수 삽입하기**: 파이프라인 중간에 직접 작성한 Python 함수를 끼워넣어 전처리/후처리를 수행하는 법을 압니다.

## 전체 흐름도

이 노트북의 전체 흐름을 한눈에 보면 이렇습니다:

```
┌─────────────────────────────────────────────────────────────┐
│                                                             │
│  [1] .env 로드 + llm 초기화 + StrOutputParser import        │
│       ↓                                                     │
│  [2] 기본 체인: prompt | llm | parser                       │
│       → invoke() 단일 실행                                  │
│       → stream() 스트리밍 실행                              │
│       → batch() 배치 실행                                   │
│       ↓                                                     │
│  [3] RunnablePassthrough                                    │
│       → 원문 보존하며 요약 결과 함께 반환                   │
│       ↓                                                     │
│  [4] RunnableParallel                                       │
│       → 요약 체인 + 키워드 체인 병렬 실행                   │
│       → 순차 vs 병렬 시간 비교                              │
│       ↓                                                     │
│  [5] RunnableLambda                                         │
│       → 전처리 함수 + 후처리 함수 체인에 삽입               │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

> **팁**: 셀을 순서대로 실행하세요! 위에서 아래로 하나씩 `Shift + Enter`를 눌러 실행하면 됩니다.

---

## 섹션 2: LCEL이란?

### 공장 컨베이어 벨트 비유

자동차 공장의 컨베이어 벨트를 상상해보세요.
차체가 들어오면 → 도색 → 조립 → 검사 → 완성차 출고.
각 공정(컴포넌트)은 독립적이지만, 벨트(파이프)로 연결되어 하나의 흐름을 만듭니다.

**LCEL(LangChain Expression Language)**은 이 컨베이어 벨트 역할을 `|` 연산자로 표현합니다:

| 방식 | 코드 | 문제점 |
|------|------|--------|
| 모듈 02 방식 | `filled = prompt.invoke(...)` / `response = llm.invoke(filled)` | 단계가 늘어날수록 코드가 길어짐 |
| **LCEL 방식** | `chain = prompt \| llm \| parser` | 한 줄로 연결, 실행은 `chain.invoke(...)` |

### Runnable 인터페이스

`|`로 연결할 수 있는 모든 컴포넌트는 **Runnable** 인터페이스를 구현합니다.
Runnable이면 세 가지 방법으로 실행할 수 있습니다:

| 메서드 | 설명 | 사용 상황 |
|--------|------|----------|
| `.invoke(input)` | 단일 입력 → 단일 출력 | 일반적인 1회 실행 |
| `.stream(input)` | 단일 입력 → 청크 단위 스트리밍 출력 | 실시간 타이핑 효과 |
| `.batch([input1, input2, ...])` | 여러 입력 → 여러 출력 (병렬 처리) | 대량 처리 |

`ChatPromptTemplate`, `ChatGoogleGenerativeAI`, `StrOutputParser` 모두 Runnable입니다.

In [1]:
# .env 로드 + llm 초기화 + StrOutputParser import
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser

# 프로젝트 루트의 .env 파일 로드 (이 노트북은 03_lcel/ 폴더에 있음)
notebook_dir = os.path.dirname(os.path.abspath('__file__'))
project_root = os.path.dirname(notebook_dir)
env_path = os.path.join(project_root, '.env')

loaded = load_dotenv(env_path)

# API 키 확인
api_key = os.getenv('GOOGLE_API_KEY')
if api_key and api_key != 'your_api_key_here':
    print(f'[OK] GOOGLE_API_KEY 로드 성공: {api_key[:10]}...')
else:
    print('[FAIL] API 키가 없습니다. 모듈 00을 먼저 완료하세요.')

# 모델 초기화
llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    temperature=0.7,
)

# StrOutputParser: AIMessage.content (str)를 꺼내주는 파서
parser = StrOutputParser()

print()
print('[OK] ChatGoogleGenerativeAI 초기화 완료!')
print(f'모델: {llm.model}')
print(f'[OK] StrOutputParser 준비 완료: {type(parser).__name__}')

[OK] GOOGLE_API_KEY 로드 성공: AIzaSyAlrJ...

[OK] ChatGoogleGenerativeAI 초기화 완료!
모델: gemini-2.5-flash
[OK] StrOutputParser 준비 완료: StrOutputParser


---

## 섹션 3: 기본 체인 - prompt | llm | parser

### | 파이프 연산자 - 레고 블록 연결 비유

레고 블록을 연결하듯이, `|` 연산자로 Runnable 컴포넌트를 이어붙입니다.
앞 블록의 출력이 자동으로 다음 블록의 입력이 됩니다:

```
dict input
    ↓
[ChatPromptTemplate]  → dict → ChatPromptValue (메시지 리스트)
    ↓
[ChatGoogleGenerativeAI] → ChatPromptValue → AIMessage
    ↓
[StrOutputParser]     → AIMessage → str
    ↓
str output
```

### StrOutputParser의 역할

모듈 02에서는 `response.content`로 직접 문자열을 꺼냈습니다.
`StrOutputParser`는 이 작업을 체인 안에서 자동으로 처리합니다:

| 모듈 02 방식 | LCEL + StrOutputParser |
|-------------|------------------------|
| `response = llm.invoke(...)` → `response.content` | `chain = prompt \| llm \| parser` → 결과가 바로 `str` |
| 타입: `AIMessage` | 타입: `str` |

In [2]:
# 기본 체인 구성 + 타입 확인
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 창의적인 작가입니다.'),
    ('human', '{topic}에 대한 짧은 시를 한국어로 써주세요. 4줄로 작성하세요.'),
])
parser = StrOutputParser()
chain = prompt | llm | parser

print(f'chain 타입: {type(chain).__name__}')
print(f'구성 단계 수: {len(chain.steps)}')
for i, step in enumerate(chain.steps):
    print(f'  [{i+1}] {type(step).__name__}')

chain 타입: RunnableSequence
구성 단계 수: 3
  [1] ChatPromptTemplate
  [2] ChatGoogleGenerativeAI
  [3] StrOutputParser


In [3]:
# invoke() - 단일 실행, 결과 타입이 str임을 확인
result = chain.invoke({'topic': '봄비'})

print('=== invoke() 결과 ===')
print(f'결과 타입: {type(result).__name__}')  # str 이어야 함
print()
print(result)

=== invoke() 결과 ===
결과 타입: TextAccessor

창밖엔 보슬보슬 봄비 내리고,
메마른 가지마다 새 숨결 스미네.
흙 내음 촉촉이, 풀잎 춤추고,
세상은 연둣빛 꿈으로 물드네.


In [4]:
# stream() - 스트리밍 실시간 출력 + 청크 수 카운트
print('=== stream() 결과 ===')
print('(텍스트가 조금씩 출력됩니다)')
print()

chunk_count = 0
for chunk in chain.stream({'topic': '가을 단풍'}):
    print(chunk, end='', flush=True)
    chunk_count += 1

print()
print()
print(f'총 청크 수: {chunk_count}개')
print('(각 청크는 단어 또는 문자 단위로 순서대로 도착합니다)')

=== stream() 결과 ===
(텍스트가 조금씩 출력됩니다)

산마다 타오르는 붉은 물결,
불꽃처럼 번져가는 고운 색깔.
바람 실어 흩날리는 붉은 춤,
내 마음마저 붉게 물들이네.

총 청크 수: 3개
(각 청크는 단어 또는 문자 단위로 순서대로 도착합니다)


In [5]:
# batch() - 여러 입력 한 번에 처리 + 결과 출력
topics = [
    {'topic': '겨울 눈'},
    {'topic': '여름 바다'},
    {'topic': '새벽 안개'},
]

results = chain.batch(topics)

print('=== batch() 결과 ===')
print(f'입력 수: {len(topics)}개 → 결과 수: {len(results)}개')
print()
for topic, result in zip(topics, results):
    print(f'[{topic["topic"]}]')
    print(result)
    print()

=== batch() 결과 ===
입력 수: 3개 → 결과 수: 3개

[겨울 눈]
하늘에서 사뿐히 내려와
세상을 하얗게 감싸네
고요한 밤 은빛으로 반짝이며
겨울의 꿈을 속삭인다

[여름 바다]
푸른 물결 넘실대는 여름 바다,
햇살 부서져 은빛으로 반짝이네.
시원한 바람 살갗 스쳐 가고,
파도 소리 마음에 평화 안기네.

[새벽 안개]
새벽 공기 가르며, 고요히 내려앉은
은빛 실타래, 세상 덮는 희미한 베일.
잠든 풍경 속, 신비로운 속삭임
햇살에 스러질, 덧없는 꿈 같아라.



---

## 섹션 4: RunnablePassthrough

### 투명 파이프 비유

공장 컨베이어 벨트에 **투명한 관**이 하나 있다고 생각해보세요.
제품이 이 관을 통과할 때 아무 변형도 없이 그대로 통과합니다.
하지만 이 관 덕분에 원본 제품이 마지막까지 보존됩니다.

`RunnablePassthrough`는 입력을 **아무 변형 없이** 다음 단계로 그대로 전달합니다.

### 언제 쓰나요?

`RunnableParallel`과 함께 사용해서, 요약/변환 결과와 동시에 **원본 데이터도 함께** 가져올 때 씁니다:

```python
RunnableParallel({
    '원문': RunnablePassthrough(),  # 입력을 그대로 통과시킴
    '요약': summary_chain,          # 입력을 요약해서 반환
})
```

| 키 | 처리 | 결과 |
|----|------|------|
| `원문` | 입력 그대로 통과 | `{'text': '원본 텍스트...'}` |
| `요약` | summary_chain 실행 | `'한 문장 요약 결과'` |

In [6]:
# RunnablePassthrough로 원본 text를 보존하며 요약 결과와 함께 반환
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

summary_prompt = ChatPromptTemplate.from_messages([
    ('human', '다음 텍스트를 한 문장으로 요약하세요:\n\n{text}'),
])
summary_chain = summary_prompt | llm | parser

# 원문 보존하며 요약 결과도 함께 반환
chain_with_passthrough = RunnableParallel({
    '원문': RunnablePassthrough(),
    '요약': summary_chain,
})

sample_text = (
    'LangChain은 대형 언어 모델(LLM)을 활용한 애플리케이션 개발을 위한 프레임워크입니다. '
    '프롬프트 관리, 체인 구성, 메모리, 에이전트 등 다양한 기능을 제공하며, '
    'OpenAI, Google Gemini, Anthropic Claude 등 다양한 LLM과 통합됩니다.'
)

result = chain_with_passthrough.invoke({'text': sample_text})

print('=== RunnablePassthrough 결과 ===')
print(f'결과 타입: {type(result).__name__}')  # dict 이어야 함
print(f'결과 키: {list(result.keys())}')
print()
print(f"원문 타입: {type(result['원문']).__name__}")
print(f"원문 (처음 50자): {result['원문']['text'][:50]}...")
print()
print(f"요약 타입: {type(result['요약']).__name__}")
print(f"요약: {result['요약']}")

=== RunnablePassthrough 결과 ===
결과 타입: dict
결과 키: ['원문', '요약']

원문 타입: dict
원문 (처음 50자): LangChain은 대형 언어 모델(LLM)을 활용한 애플리케이션 개발을 위한 프레임워크입...

요약 타입: TextAccessor
요약: LangChain은 대형 언어 모델(LLM) 기반 애플리케이션 개발을 위한 프레임워크로, 프롬프트 관리, 체인 구성 등의 다양한 기능과 여러 LLM과의 통합을 제공합니다.


---

## 섹션 5: RunnableParallel - 병렬 실행

### 주방 여러 요리사 비유

레스토랑 주방에 요리사가 한 명뿐이라면 전채요리 → 메인요리 → 디저트를 **순서대로** 만들어야 합니다.
하지만 요리사가 세 명이라면 세 가지를 **동시에** 만들 수 있어서 훨씬 빠릅니다.

`RunnableParallel`은 여러 체인을 **동시에(병렬로)** 실행합니다:

```python
RunnableParallel({
    '요약':   summary_chain,   # 요리사 1: 요약
    '키워드': keyword_chain,   # 요리사 2: 키워드 추출
})
```

### 결과 형태

결과는 **딕셔너리**로 반환됩니다. 키는 `RunnableParallel`에 지정한 이름입니다:

```python
result = {
    '요약':   '한 문장 요약 결과',
    '키워드': '키워드1, 키워드2, 키워드3',
}
```

### 순차 실행 vs 병렬 실행

| 방식 | 소요 시간 | 설명 |
|------|-----------|------|
| 순차 실행 | 요약 시간 + 키워드 시간 | 하나씩 기다림 |
| **병렬 실행** | max(요약 시간, 키워드 시간) | 동시에 실행 |

In [7]:
# 요약 체인 + 키워드 추출 체인 각각 단독 정의
keyword_prompt = ChatPromptTemplate.from_messages([
    ('human', '다음 텍스트에서 핵심 키워드 3개를 쉼표로 구분해 추출하세요:\n\n{text}'),
])
keyword_chain = keyword_prompt | llm | parser

# (summary_chain은 셀 12에서 이미 정의됨)
text = (
    'LangChain은 대형 언어 모델(LLM)을 활용한 애플리케이션 개발을 위한 프레임워크입니다. '
    '프롬프트 관리, 체인 구성, 메모리, 에이전트 등 다양한 기능을 제공하며, '
    'OpenAI, Google Gemini, Anthropic Claude 등 다양한 LLM과 통합됩니다.'
)

print('=== 각 체인 단독 실행 확인 ===')
s = summary_chain.invoke({'text': text})
k = keyword_chain.invoke({'text': text})
print(f'요약: {s}')
print(f'키워드: {k}')

=== 각 체인 단독 실행 확인 ===
요약: LangChain은 프롬프트 관리, 체인 구성 등 다양한 기능을 제공하고 여러 LLM과 통합하여 대형 언어 모델(LLM) 기반 애플리케이션 개발을 지원하는 프레임워크입니다.
키워드: LangChain, LLM, 프레임워크


In [8]:
# RunnableParallel로 두 체인 병렬 실행 → 딕셔너리 결과 확인
parallel_chain = RunnableParallel({
    '원문': RunnablePassthrough(),
    '요약': summary_chain,
    '키워드': keyword_chain,
})

result = parallel_chain.invoke({'text': text})

print('=== RunnableParallel 결과 ===')
print(f'결과 타입: {type(result).__name__}')
print(f'결과 키: {list(result.keys())}')
print()
print(f"원문 (처음 40자): {result['원문']['text'][:40]}...")
print(f"요약: {result['요약']}")
print(f"키워드: {result['키워드']}")

=== RunnableParallel 결과 ===
결과 타입: dict
결과 키: ['원문', '요약', '키워드']

원문 (처음 40자): LangChain은 대형 언어 모델(LLM)을 활용한 애플리케이션 개발을...
요약: LangChain은 대형 언어 모델(LLM) 기반 애플리케이션 개발을 위한 프레임워크로, 프롬프트 관리, 체인 구성 등 다양한 기능과 여러 LLM과의 통합을 지원합니다.
키워드: LangChain, LLM, 프레임워크


In [9]:
# 순차 실행 vs 병렬 실행 시간 비교 (time 모듈)
import time

# 순차 실행
start = time.time()
s1 = summary_chain.invoke({'text': text})
k1 = keyword_chain.invoke({'text': text})
seq_time = time.time() - start

# 병렬 실행
start = time.time()
result_parallel = parallel_chain.invoke({'text': text})
par_time = time.time() - start

print('=== 순차 실행 vs 병렬 실행 시간 비교 ===')
print(f'순차 실행: {seq_time:.2f}초  (요약 + 키워드를 하나씩)')
print(f'병렬 실행: {par_time:.2f}초  (요약 + 키워드를 동시에)')
print()
if par_time < seq_time:
    ratio = seq_time / par_time
    print(f'[OK] 병렬 실행이 {ratio:.1f}배 빠릅니다!')
else:
    print('[FAIL] 네트워크 상황에 따라 차이가 없을 수도 있습니다. 재실행해보세요.')

=== 순차 실행 vs 병렬 실행 시간 비교 ===
순차 실행: 8.36초  (요약 + 키워드를 하나씩)
병렬 실행: 5.12초  (요약 + 키워드를 동시에)

[OK] 병렬 실행이 1.6배 빠릅니다!


---

## 섹션 6: RunnableLambda - 커스텀 함수

### 파이프라인 중간에 직접 손 넣기 비유

공장 컨베이어 벨트가 돌아가는 중간에, 검사원이 직접 손을 뻗어 제품을 조정하는 장면을 상상해보세요.
벨트를 멈추지 않고도 중간에 원하는 처리를 추가할 수 있습니다.

`RunnableLambda`는 **일반 Python 함수를 Runnable로 변환**해서 체인 어디에든 끼워넣을 수 있게 합니다:

```python
from langchain_core.runnables import RunnableLambda

def my_function(inputs):
    # 원하는 처리
    return processed_inputs

chain = RunnableLambda(my_function) | prompt | llm | parser | RunnableLambda(postprocess)
#       ^^^^^^^^^^^^^^^^^^^^^^^^^                                ^^^^^^^^^^^^^^^^^^^^^^^^^
#       전처리 (입력 정리)                                       후처리 (결과 포맷)
```

### 활용 예시

| 위치 | 용도 | 예시 |
|------|------|------|
| 체인 앞 (전처리) | 입력 정규화 | 공백 제거, 소문자 변환, 검증 |
| 체인 중간 | 데이터 변환 | 형식 변환, 필터링 |
| 체인 끝 (후처리) | 결과 포맷 | 번호 붙이기, 마크다운 추가 |

In [10]:
# RunnableLambda로 전처리(입력 정리) + 후처리(결과 포맷) 체인 구성
from langchain_core.runnables import RunnableLambda

def preprocess(inputs: dict) -> dict:
    """앞뒤 공백 제거 + 소문자 변환"""
    cleaned = inputs['topic'].strip().lower()
    print(f'  [전처리] "{inputs["topic"]}" → "{cleaned}"')
    return {'topic': cleaned}

def postprocess(text: str) -> str:
    """결과에 줄 번호 추가"""
    lines = text.strip().split('\n')
    numbered = '\n'.join(
        f'{i+1}. {line}'
        for i, line in enumerate(lines)
        if line.strip()
    )
    return numbered

chain_with_lambda = (
    RunnableLambda(preprocess)
    | prompt
    | llm
    | parser
    | RunnableLambda(postprocess)
)

print('=== RunnableLambda 결과 ===')
print()
# 공백이 있는 입력도 전처리에서 정리됨
result = chain_with_lambda.invoke({'topic': '  SPRING RAIN  '})
print()
print('[후처리 결과 - 줄 번호 추가]')
print(result)

=== RunnableLambda 결과 ===

  [전처리] "  SPRING RAIN  " → "spring rain"

[후처리 결과 - 줄 번호 추가]
1. 봄비 가만히 내려와
2. 얼어붙던 땅을 녹이고
3. 파릇한 새싹 돋아나
4. 생명의 숨결 피어나네.


---

## 섹션 7: 마무리

### 배운 것 정리

#### invoke / stream / batch 비교표

| 메서드 | 입력 | 출력 | 사용 상황 |
|--------|------|------|----------|
| `invoke(input)` | 딕셔너리 1개 | 결과 1개 | 일반적인 단일 실행 |
| `stream(input)` | 딕셔너리 1개 | 청크 스트림 | 실시간 타이핑 효과 |
| `batch([i1, i2, ...])` | 딕셔너리 리스트 | 결과 리스트 | 여러 입력 대량 처리 |

#### 핵심 컴포넌트 정리

| 컴포넌트 | 역할 | 임포트 경로 |
|---------|------|-------------|
| `StrOutputParser` | AIMessage → str 변환 | `langchain_core.output_parsers` |
| `RunnablePassthrough` | 입력을 변형 없이 통과 | `langchain_core.runnables` |
| `RunnableParallel` | 여러 체인 동시 실행 → dict 반환 | `langchain_core.runnables` |
| `RunnableLambda` | 일반 Python 함수를 Runnable로 변환 | `langchain_core.runnables` |

#### 핵심 코드 패턴

```python
# 1. 기본 체인
chain = prompt | llm | parser
result = chain.invoke({'topic': '봄비'})          # str 반환

# 2. 병렬 실행 + 원문 보존
parallel = RunnableParallel({
    '원문':   RunnablePassthrough(),
    '요약':   summary_chain,
    '키워드': keyword_chain,
})
result = parallel.invoke({'text': '...'})          # dict 반환

# 3. 커스텀 함수 삽입
chain = RunnableLambda(preprocess) | prompt | llm | parser | RunnableLambda(postprocess)
```

## 다음 모듈 예고: 모듈 04 - 출력 파서

모듈 03에서는 `StrOutputParser`로 AIMessage를 문자열로 꺼냈습니다.

다음 모듈에서는 LLM의 응답을 **구조화된 형태(JSON, Python 객체)**로 꺼내는 방법을 배웁니다:

```python
# 모듈 04 미리보기
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

class Recipe(BaseModel):
    name: str
    ingredients: list[str]
    cooking_time: int

parser = JsonOutputParser(pydantic_object=Recipe)
chain = prompt | llm | parser
result = chain.invoke({'dish': '김치찌개'})  # → dict 반환
print(result['name'])                         # '김치찌개'
```

### 준비 체크리스트

다음 모듈로 넘어가기 전에 확인하세요:

- [ ] 셀 8: invoke() 결과가 `str` 타입으로 출력되었나요?
- [ ] 셀 9: stream()으로 텍스트가 조금씩 출력되고 청크 수가 표시되었나요?
- [ ] 셀 10: batch()로 3개 주제의 시가 각각 출력되었나요?
- [ ] 셀 15: 병렬 실행 결과가 딕셔너리(`원문`, `요약`, `키워드` 키)로 반환되었나요?
- [ ] 셀 16: 병렬 실행 시간이 순차 실행보다 빠르게 나왔나요?
- [ ] 셀 18: RunnableLambda로 전/후처리된 결과에 줄 번호가 붙어서 출력되었나요?

모두 체크했다면 `04_output_parsers/모듈04_출력_파서.ipynb`를 열어보세요!

---

수고하셨습니다!